In [1]:
%pip install xgboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 23.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 30.0 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 19.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [xgboost]m2/3 [xgboost]ccl-cu13]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


1. Importing Libraries required

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

In [5]:
from sklearn.cluster import KMeans

2. Data Loading and Understanding

In [6]:
%pip install pandas openpyxl


Note: you may need to restart the kernel to use updated packages.


In [7]:
df = pd.read_excel('data/Telco_customer_churn.xlsx')

FileNotFoundError: [Errno 2] No such file or directory: 'data/Telco_customer_churn.xlsx'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
pd.set_option("display.max_columns",None)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#dropping customerID as its not required for modelling 
df=df.drop(columns=["CustomerID"])

In [ ]:
df.head(2)

In [ ]:
#printing unbique values from all the columns
for col in df.columns:
    print(col, df[col].unique())
    print("-"*50)

In [ ]:
print(df["Senior Citizen"].unique())

In [ ]:
df.isnull().sum()

In [ ]:
#dropping Churn Reason Column as it would lead to target leakage during modelling 
df = df.drop(columns=["Churn Reason"])

In [ ]:
df.head(2)

In [ ]:
print(df["Churn Label"].value_counts())

** Insights **
1. CustomerID and Churn Reason is dropped as its not useful for modelling
2. No missing values found
3. Class Imbalance found in target column i.e. Churn Label 

In [ ]:
df.describe()

In [ ]:
len(df[df['Total Charges']==" "])

In [ ]:
df[df['Total Charges']==" "]

In [ ]:
#replacing empty spaces in Total Charges Column to 0.0 before chnaging data type
df['Total Charges']= df['Total Charges'].replace(" ","0.0")

In [ ]:
df['Total Charges'] = df['Total Charges'].astype(float)

In [ ]:
df.describe()

In [ ]:
df.info()

*********************
Lets do EDA
histogram to chk distribution
boxplot to identify outliers
*********************

In [ ]:
def plot_histogram(df,column_name):
    plt.figure(figsize=(5,3))
    sns.histplot(df[column_name],kde=True)
    plt.title(f"Distribution of {column_name}")

    #calculate the median and mean for column name
    col_mean = df[column_name].mean()
    col_median = df[column_name].median()

   #draw verticaal lines for mean and median 
    plt.axvline(col_mean, color="red", linestyle="--",label="Mean")
    plt.axvline(col_median, color="green", linestyle="-",label="Median")

    plt.legend()
    plt.show()

In [ ]:
df.columns


In [ ]:
plot_histogram(df,'Tenure Months')

In [ ]:
plot_histogram(df,'Monthly Charges')

In [ ]:
plot_histogram(df,'Total Charges')

In [ ]:
def box_plot(df,column_name):
    plt.figure(figsize = (5,3))
    plt.title(f"Box Plot for {column_name}")
    sns.boxplot(y=df[column_name])
    plt.ylabel(column_name)
    plt.show()

In [ ]:
box_plot(df,"Tenure Months")

In [ ]:
box_plot(df,"Monthly Charges")

In [ ]:
box_plot(df,"Total Charges")

******************* No Outliers Identified in these columns***************************

In [ ]:
#making heatmap to chk correlation of 3 main numerical columns
plt.figure(figsize=(8,4))
sns.heatmap(df[["Tenure Months","Monthly Charges", "Total Charges"]].corr(),annot=True,cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

********** Aanalyzing Categorical Data using countplot************

In [ ]:
object_cols = df.select_dtypes(include="object").columns.to_list()

In [ ]:
object_cols = object_cols + ["Senior Citizen"]
object_cols

In [ ]:
for col in object_cols:
    plt.figure(figsize=(5,3))
    plt.title(col)
    sns.countplot(x=df[col])
    plt.show()
    

******************************************************

Data Preprocessing 

******************************************************


In [ ]:
print(df["Churn Value"].value_counts())

In [ ]:
#Label Encoding of Categorical Features
obj_cols= df.select_dtypes(include="object").columns
print(obj_cols)

In [ ]:
df.info()

In [ ]:
#initialize dictionary to save encoders for all objects datatype columns
encoders = {}

#apply label encoding and save the encoders
for column in obj_cols:
    labelencoder=LabelEncoder()
    df[column]=labelencoder.fit_transform(df[column])
    encoders[column]=labelencoder
    
#save encoders in pickle file
with open("encoders.pkl","wb") as f:
    pickle.dump(encoders,f)
    

In [ ]:
encoders

In [ ]:
df.head()

************ Training and Test Data spliting *****************
1. Dropping Churn Value to avaoid data leakage as using churn value as target column
2. Dropping Lat Long Column before processing of latitude and Longitude as the later 2 columna re cleaner to process data

In [ ]:
df.head()

In [ ]:
df=df.drop(columns=["Lat Long"]).head(2)

In [ ]:
df.head(2) # done with churn label and lat long column removal

In [ ]:
# Train/Test Split (Mandatory before Target Encoding to avoid data leakage)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['Churn Value'])